# 02 — Geometry Optimization

Generate staged optimization inputs and audit a bundled synthetic output. Replace the sample with a real `.out` after running ORCA.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


In [ ]:
from orca_reviewer.input_builder import OrcaInput

preopt = OrcaInput(
    method_line="r2SCAN-3c TightSCF Opt PAL8",
    charge=0,
    multiplicity=1,
    xyzfile="structure.xyz",
    blocks={"maxcore": ["2500"]},
    comments=["Stage 1: economical preoptimization"],
)
production = OrcaInput(
    method_line="PBE0 D4 def2-TZVP TightSCF TightOpt Opt Freq SMD(Acetonitrile) PAL8",
    charge=0,
    multiplicity=1,
    xyzfile="preopt.xyz",
    blocks={"maxcore": ["3000"], "scf": ["MaxIter 300"], "geom": ["MaxIter 250"]},
    comments=["Stage 2: production optimization and minimum verification"],
)
print(preopt.render())
print(production.render())

## Parse convergence evidence

In [ ]:
from orca_reviewer.parsers import parse_final_energy, terminated_normally
sample = ROOT / "sample_data" / "sample_opt.out"
print("Normal termination:", terminated_normally(sample))
print("Final energy / Eh:", parse_final_energy(sample))
print(sample.read_text())

## Geometry audit prompts

After a real run, answer: Did every criterion pass? Did connectivity change? Which conformer and spin state was tested? Was the stationary point verified by frequencies?